In [ ]:
from sim import run_sim
from utils import render_grid_frame_arena, make_two_rooms_with_corridor, grid
import numpy as np
import pandas as pd
import tqdm
import itertools
import os

In [ ]:
# Init
output_folder = 'runs'
history_folder = 'pkls'
os.makedirs(output_folder, exist_ok=True)
os.makedirs(os.path.join(output_folder, history_folder), exist_ok=True)

# history = {'init': {'forgetting_rates': {'M_fr' : M_forgetting_rate, 'T_fr': T_forgetting_rate, 'D_fr': D_forgetting_rate}, 
#                 'ticks': {'T_ticks': T_ticks, 'D_ticks': D_ticks},
#                 'base_scale': base_scale,
#                 'T_act_1': T_control_scales[1],
#                 'D_act_1': D_control_scales[1],
#                 'detection_threshold': danger_detection_threshold,
#                 'utils': {'M': {'agent': U_agent_base, 'shelter': U_shelter_base, 'threat': U_threat_base},
#                           'T': U_T,
#                           'D': U_D},
#                 'M_act_habits_single': E_single,
#                 'threat_loc': rightcol_states[0],
#                 'shelter_loc': np.array(leftcol_states),
#                 }, 
#        'agent_loc': [], 
#                         'M_beliefs': [], 'M_util': [],  'M_info_gain': [], 'M_q_pi': [], 'M_neg_efe': [], 'M_action': [], 
#                         'T_beliefs': [], 'T_util': [], 'T_info_gain': [], 'T_q_pi': [], 'T_neg_efe': [], 'T_act_t': [], 'T_action': [],
#                         'D_beliefs': [], 'D_util': [], 'D_info_gain': [], 'D_q_pi': [], 'D_neg_efe': [], 'D_act_t': [], 'D_action': [],}

mask, regions = make_two_rooms_with_corridor(4, 0, 3, 2, (1,2), prefer_total_cols=None)

map = grid(mask=mask)

In [ ]:
# Defining metrics
def t_social_investigation(history, arena, parts=1, d_threshold=1):
    agent_locs = history['agent_loc']
    total_t = len(agent_locs)
    threat_state = int(history['init']['threat_loc']) 
    chunk_size = total_t // parts
    
    results = []

    for i in range(parts):
        start_idx = i * chunk_size
        if i == parts - 1:
            end_idx = total_t
        else:
            end_idx = (i + 1) * chunk_size
        segment = agent_locs[start_idx:end_idx]
        investigating_t = 0
        for agent_state in segment:
            if arena.manhattan_states(agent_state, threat_state) <= d_threshold:
                investigating_t += 1
        results.append(investigating_t / len(segment))

    return results

def t_shelter(history, parts=1):
    agent_locs = history['agent_loc']
    total_t = len(agent_locs)

    shelter_states = history['init']['shelter_loc']
    
    chunk_size = total_t // parts
    results = []

    for i in range(parts):
        start_idx = i * chunk_size
        if i == parts - 1:
            end_idx = total_t
        else:
            end_idx = (i + 1) * chunk_size
        
        segment = agent_locs[start_idx:end_idx]
        shelter_t = 0

        for agent_state in segment:
            if agent_state in shelter_states:
                shelter_t += 1
        results.append(shelter_t / len(segment))
    return results

# def t_retreat():
#     return

In [ ]:
default_params = {
    'gif_path': None,
    'pkl_path': None,
    'M_fr': 0.1, 
    'D_fr': 0.1, 
    'T_fr': 0.2, 
    'max_steps': 1000, 
    'id_threshold': 0.8,
    'T_ticks': 4, 
    'D_ticks': 16, 
    'k_shelter': 0.6, 
    'k_threat': 0.8, 
    'threat_grad': [-0.1, -0.1, -0.2, -0.25], 
    'shelter_grad': [-0.1, 0.1, 0.15, 0.2, 0.3],
    'delta_stay': 0.15, 
    'epistemic_drive': 1.0, 
    'T_scale': (-0.3, -0.3), 
    'D_scale': (0.5, 0.7)
}

param_grid = {
    'M_fr': [0.1, 0.2, 0.3],
    'T_scale': [
        (-0.3, -0.3),
        (-0.5, -0.5),
        (-0.1, -0.1)
    ],
    # 'threat_grad': [
    #     [-0.1, -0.1, -0.2, -0.25],
    #     [-0.5, -0.5, -0.8, -1.0],
    #     [0.0, 0.0, 0.0, 0.0]
    # ]
}

# run_history = run_sim(gif_path=None, pkl_path=None, M_fr=0.1, D_fr=0.1, T_fr=0.2, max_steps=1000, id_threshold=0.8, \
#             T_ticks=4, D_ticks=16, k_shelter=0.6, k_threat=0.8, threat_grad=[-0.1, -0.1, -0.2, -0.25], shelter_grad=[-0.1, 0.1, 0.15, 0.2, 0.3], \
#             delta_stay=0.15, epistemic_drive=1.0, T_scale=(-0.3, -0.3), D_scale=(0.5, 0.7))

# generate all combinations from the grid
keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"Total simulations to run: {len(combinations)}")


In [ ]:
run_name = 'saturation_test'
results = []

# Iterate through combinations
for i, varying_params in tqdm.tqdm(enumerate(combinations)):
    current_run_config = default_params.copy()
    current_run_config.update(varying_params)

    current_run_config['pkl_path'] = f'runs/pkls/{run_name}_{i}.pkl'

    run_history = run_sim(**current_run_config)
    
    log_entry = varying_params.copy()

    segments = 5
    t_si = t_social_investigation(run_history, map, segments)
    t_sh = t_shelter(run_history, segments)

    for j in range(segments):
        log_entry[f'SocialInvestigation_{j}'] = t_si[j]
        log_entry[f'Shelter_{j}'] = t_sh[j]

    results.append(log_entry)

print('Simulations complete')

In [ ]:
df = pd.DataFrame(results)
print(df)

In [ ]:
df.to_csv(f"runs/{run_name}.csv", index=False)